## 1. Download training data from github

Download a CSV file containing molecular data for training.

In [ ]:
import requests

input_data="https://raw.githubusercontent.com/dgront/chem-ml/refs/heads/main/INPUTS/xlogp_JChemEdu/xlogp.tsv"
req = requests.get(input_data)
table = []
for row in req.text.splitlines():
  tokens = row.split("\t")
  if len(tokens) == 4:
    if len(tokens[2]) == 0: continue
    table.append(tokens)

print(table[0:5])
print(len(table))

[['cmpdname', 'mf', 'isosmiles', 'xlogp'], ['1-Aminopropan-2-ol', 'C3H9NO', 'CC(CN)O', '-1'], ['"1-Chloro-2,4-dinitrobenzene"', 'C6H3ClN2O4', 'C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl', '2.3'], ['9-Ethyladenine', 'C7H9N5', 'CCN1C=NC2=C(N=CN=C21)N', '0.2'], ['"1,2-Dichloroethane"', 'C2H4Cl2', 'C(CCl)Cl', '1.5']]
266712


## 2. Install dependencies

We need RDKit to process SMILES; tensorflow to prepare training data as tensors

In [ ]:
!pip install numpy rdkit tensorflow
from rdkit import Chem
import tensorflow as tf
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 22.6 MB/s eta 0:00:00


## 3. Extract features from SMILES


First we define a function that accepts a SMILES and returns data we need to train GNN model. For a molecule of N atoms and M bonds, the function returns:

*   list of element names for all atoms - N strings like "C", "Fe", etc
*   list of hybridisation for all atoms - N strings like "sp2", "sp" etc
*   list of bond types for all bonds - M tuples (int, int, str) for a bond that connects atoms given by the two indexes; the string may be "SINGLE", "DOUBLE" etc to denote the bond type
*   list of aromaticity flags for all atoms - N integers, 1 if the atom is part of an aromatic ring, 0 otherwise
*   list of formal charges for all atoms - N integers like -1, 0, +1 indicating the atom's electric charge
*   list of chirality tags for all atoms - N strings like "CHI_TETRAHEDRAL_CW", "CHI_UNSPECIFIED" etc

In [ ]:
from typing import List, Tuple

def extract_atoms_and_bonds(smiles: str) -> Tuple[List[str], List[str], List[Tuple[int, int, str]]]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Invalid SMILES string.")

    mol = Chem.AddHs(mol)
    # Extract atom list
    atom_list = []
    hybrid_list = []
    aromatic_list = []
    charge_list = []
    chirality_list = []

    for atom in mol.GetAtoms():
        atom_list.append(atom.GetSymbol())
        hybrid_list.append(str(atom.GetHybridization()))
        aromatic_list.append(int(atom.GetIsAromatic()))         # 1 if aromatic, 0 if not
        charge_list.append(atom.GetFormalCharge())              # e.g. -1, 0, +1
        chirality_list.append(str(atom.GetChiralTag()))         # e.g. 'CHI_TETRAHEDRAL_CW'

    # Extract bond list
    bond_list = []
    for bond in mol.GetBonds():
        begin_idx = bond.GetBeginAtomIdx()
        end_idx = bond.GetEndAtomIdx()
        bond_type = str(bond.GetBondType())
        is_conjugated = int(bond.GetIsConjugated())             # 1 if conjugated, 0 if not
        bond_list.append((begin_idx, end_idx, bond_type, is_conjugated))

    return atom_list, hybrid_list, aromatic_list, charge_list, chirality_list, bond_list

## 4. Define a list of elements and bond types that will be explicitely encoded in one-hot manner.

Here we list only most popular elements. Every element that is not on the list, will be encoded as *unknown*.

Aromaticity and formal charge don't need a vocabulary here because they're already numbers (0/1 and -1/0/+1)

In [ ]:
allowed_bond_types    = ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC']
allowed_elements      = ['C', 'N', 'O', 'H', 'Cl', 'P', 'S', 'As', 'Br', 'Ca', 'Se', 'I', 'K', 'Mg', 'Na', 'Ni', 'W', 'F', 'B', 'Hg', 'Al', 'Li', 'Zn', 'Si', 'Co', 'Pb', 'Sn', 'Cu', 'Ba', 'Fe', 'Mn', 'Cr']
allowed_hybridization = ['SP', 'SP2', 'SP3', 'SP3D', 'SP3D2', 'S', 'UNSPECIFIED']
allowed_chirality     = ['CHI_UNSPECIFIED', 'CHI_TETRAHEDRAL_CW', 'CHI_TETRAHEDRAL_CCW', 'CHI_OTHER']  # NEW

allowed_bond_indexed         = {token: idx for idx, token in enumerate(allowed_bond_types)}
allowed_elements_indexed     = {token: idx for idx, token in enumerate(allowed_elements)}
allowed_hybridization_indexed = {token: idx for idx, token in enumerate(allowed_hybridization)}
allowed_chirality_indexed    = {token: idx for idx, token in enumerate(allowed_chirality)}  # NEW

## 5. One-hot encoding function

A function that provides **one-hot encoding of a string** by finding its position on a list of K allowed strings. If not found, the given `input_string` is encoded as K+1st variant

In [ ]:
import numpy as np

def one_hot_encode(allowed_tokens, input_string):
    """
    One-hot encode a string based on a given dict of string tokens.

    Returns:
        np.ndarray: A 1D numpy array representing the one-hot encoded token.
    """

    # Initialize an empty matrix
    one_hot_matrix = np.zeros((len(allowed_tokens) + 1), dtype=int)

    # Fill in the one-hot matrix
    if input_string in allowed_tokens:
        one_hot_matrix[allowed_tokens[input_string]] = 1
    else:
        one_hot_matrix[len(allowed_tokens)] = 1

    return one_hot_matrix

# ---- a short test
print(one_hot_encode(allowed_elements_indexed, 'Cl'))
print(one_hot_encode(allowed_elements_indexed, 'C'))
print(one_hot_encode(allowed_bond_indexed, 'DOUBLE'))
print(one_hot_encode(allowed_chirality_indexed, 'CHI_TETRAHEDRAL_CW'))

[0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0]
[0 1 0 0 0]


## 6. Feature dimensions test

In [ ]:
#run for just one molecule to verify dimemsions
test_name, test_formula, test_smiles, test_logp = table[1]
atoms, hybridization, aromatic, charge, chirality, bonds = extract_atoms_and_bonds(test_smiles)

encoded_elements  = tf.stack([one_hot_encode(allowed_elements_indexed, a) for a in atoms])
encoded_hybrid    = tf.stack([one_hot_encode(allowed_hybridization_indexed, h) for h in hybridization])
encoded_chirality = tf.stack([one_hot_encode(allowed_chirality_indexed, c) for c in chirality])
encoded_aromatic  = tf.constant([[a] for a in aromatic], dtype=tf.int32)
encoded_charge    = tf.constant([[c] for c in charge], dtype=tf.int32)

print(f"Molecule     : {test_name} | {test_smiles}")
print(f"Num atoms    : {len(atoms)}")
print()
print(f"elements     : {encoded_elements.shape}")
print(f"hybridization: {encoded_hybrid.shape}")
print(f"chirality    : {encoded_chirality.shape}")
print(f"aromatic     : {encoded_aromatic.shape}")
print(f"charge       : {encoded_charge.shape}")

encoded_atoms = tf.concat([encoded_elements, encoded_hybrid, encoded_chirality, encoded_aromatic, encoded_charge], axis=1)
print()
print(f"final atoms  : {encoded_atoms.shape}")

Molecule     : 1-Aminopropan-2-ol | CC(CN)O
Num atoms    : 14

elements     : (14, 33)
hybridization: (14, 8)
chirality    : (14, 5)
aromatic     : (14, 1)
charge       : (14, 1)

final atoms  : (14, 48)


## 7. Load the SMILES dataset

process all molecules parallely using multiple CPU cores

In [ ]:
import numpy as np
from multiprocessing import Pool, cpu_count

def one_hot_encode_np(allowed_tokens, input_string):
    """Pure numpy version of one_hot_encode, safe for multiprocessing"""
    one_hot = np.zeros(len(allowed_tokens) + 1, dtype=np.int32)
    if input_string in allowed_tokens:
        one_hot[allowed_tokens[input_string]] = 1
    else:
        one_hot[-1] = 1
    return one_hot

# This function runs in a separate CPU process for each molecule.
def process_molecule(row):
    """Process a single molecule - runs in parallel"""
    name, formula, smiles, logp = row

    try:
        atoms, hybridization, aromatic, charge, chirality, bonds = extract_atoms_and_bonds(smiles)
        if len(atoms) < 3 or len(bonds) < 2:
            return None
    except:
        return None
     # parse XLogP value — skip if NULL or malformed
    try:
        logp_flt = float(logp)
    except:
        return None

    # atom features
    encoded_elements  = np.stack([one_hot_encode_np(allowed_elements_indexed, a) for a in atoms])
    encoded_hybrid    = np.stack([one_hot_encode_np(allowed_hybridization_indexed, h) for h in hybridization])
    encoded_chirality = np.stack([one_hot_encode_np(allowed_chirality_indexed, c) for c in chirality])
    encoded_aromatic  = np.array([[a] for a in aromatic], dtype=np.int32)
    encoded_charge    = np.array([[c] for c in charge], dtype=np.int32)

    encoded_atoms = np.concatenate([
        encoded_elements,
        encoded_hybrid,
        encoded_chirality,
        encoded_aromatic,
        encoded_charge,
    ], axis=1)

    # bond features - pure numpy
    indexing = np.array([(bi, bj) for (bi, bj, _t, _c) in bonds], dtype=np.int32).T
    encoded_bonds = np.stack([
        np.concatenate([
            one_hot_encode_np(allowed_bond_indexed, b[2]),
            np.array([b[3]], dtype=np.int32)
        ], axis=0)
        for b in bonds
    ])

    return {
        'atoms':    encoded_atoms,
        'indexing': indexing,
        'bonds':    encoded_bonds,
        'logp':     logp_flt,
        'name':     name,
        'formula':  formula
    }

#progress bar
from tqdm import tqdm

#decide how many molecules to precess
rows = table[1:100000]
print(f"Processing {len(rows)} molecules using {cpu_count()} CPU cores...")

# distributes rows across all CPU cores
with Pool(cpu_count()) as pool:
    results = list(tqdm(
        pool.imap(process_molecule, rows),
        total=len(rows),
        desc="Processing molecules"
    ))

# --- collect results ---
V_features = []
E_indexing = []
E_features = []
Y_labels   = []
names      = []
formulas   = []

for r in results:
    if r is None:
        continue
    V_features.append(tf.constant(r['atoms'],    dtype=tf.int32))
    E_indexing.append(tf.constant(r['indexing'], dtype=tf.int32))
    E_features.append(tf.constant(r['bonds'],    dtype=tf.int32))
    Y_labels.append(tf.constant([r['logp']],     dtype=tf.float32))
    names.append(r['name'])
    formulas.append(r['formula'])

print(f"Successfully processed: {len(V_features)} molecules")

Processing 99999 molecules using 2 CPU cores...


Processing molecules:  10%|█         | 10039/99999 [00:14<01:32, 971.86it/s][12:22:08] Explicit valence for atom # 1 Br, 3, is greater than permitted
[12:22:08] Explicit valence for atom # 1 Br, 5, is greater than permitted
[12:22:08] Explicit valence for atom # 1 Cl, 3, is greater than permitted
Processing molecules:  11%|█         | 11075/99999 [00:16<01:58, 753.29it/s][12:22:09] WARNING: not removing hydrogen atom without neighbors
[12:22:09] WARNING: not removing hydrogen atom without neighbors
Processing molecules:  13%|█▎        | 13418/99999 [00:19<01:50, 782.29it/s][12:22:13] Explicit valence for atom # 1 Cl, 5, is greater than permitted
[12:22:13] WARNING: not removing hydrogen atom without neighbors
[12:22:13] WARNING: not removing hydrogen atom without neighbors
Processing molecules:  23%|██▎       | 22892/99999 [00:34<02:50, 453.18it/s][12:22:28] WARNING: not removing hydrogen atom without neighbors
[12:22:28] WARNING: not removing hydrogen atom without neighbors
Processing

KeyboardInterrupt: 

In [ ]:
# ---- a short test
print(formulas[10])
print(V_features[10])
print(E_features[10])
print(E_indexing[10])

C2H3ClO
tf.Tensor(
[[1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 1 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0
  0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 1 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 1 0 1 0 0 0 0 0 0]], shape=(7, 48), dtype=int32)
tf.Tensor(
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]
 [1 0 0 0 0 0]], shape=(6, 6), dtype=int32)
tf.Tensor(
[[0 1 0 0 0 1]
 [1 2 3 4 5 6]], shape=(2, 6), dtype=int32)


## 8. Prepare training / validation / test data sets

All the molecules are split in 60%, 20%, 20% ratio

In [ ]:
import random

# Define the split ratios
train_ratio = 0.6
val_ratio = 0.2
test_ratio = 0.2

# Calculate the split indices
data_len = len(V_features)
train_split = int(train_ratio * data_len)
val_split = int((train_ratio + val_ratio) * data_len)

# Generate shuffled indices
indices = list(range(data_len))
random.shuffle(indices)

# Split the lists using the shuffled indices
V_train = [V_features[i] for i in indices[:train_split]]
V_val = [V_features[i] for i in indices[train_split:val_split]]
V_test = [V_features[i] for i in indices[val_split:]]

E_i_train = [E_indexing[i] for i in indices[:train_split]]
E_i_val = [E_indexing[i] for i in indices[train_split:val_split]]
E_i_test = [E_indexing[i] for i in indices[val_split:]]

E_f_train = [E_features[i] for i in indices[:train_split]]
E_f_val = [E_features[i] for i in indices[train_split:val_split]]
E_f_test = [E_features[i] for i in indices[val_split:]]

Y_train = [Y_labels[i] for i in indices[:train_split]]
Y_val = [Y_labels[i] for i in indices[train_split:val_split]]
Y_test = [Y_labels[i] for i in indices[val_split:]]

names_train = [names[i] for i in indices[:train_split]]
names_val = [names[i] for i in indices[train_split:val_split]]
names_test = [names[i] for i in indices[val_split:]]

formulas_train = [formulas[i] for i in indices[:train_split]]
formulas_val = [formulas[i] for i in indices[train_split:val_split]]
formulas_test = [formulas[i] for i in indices[val_split:]]

print(f"Total molecules : {data_len}")
print(f"Train           : {len(V_train)}")
print(f"Validation      : {len(V_val)}")
print(f"Test            : {len(V_test)}")

Total molecules : 91687
Train           : 55012
Validation      : 18337
Test            : 18338
